In [1]:
%matplotlib tk
from equipment.equipment_init import *
from project.sc8583 import project

# if "ic" in globals():
#     ic.close()

# ic = project(device="sc8583", revision="1p1", emulator="cp2112", logging=False, is_gui=False)

# from phy.power_z import km003c
# from time import sleep as delay
# from interface.cui_logger import logger as log
# uart_ret = log.scan_uart()


# if uart_ret is not None:
#     for key, value in uart_ret.items():
#         if uart_ret[key]["pid"] == 0xEA60:
#             com_port = key
#             pz = km003c(port="COM16")

initialized the dm6500 connection
initialized the keysight n6705 connection
initialized the scope connection
failed to initialize asd-906b
initialized the it8511a connection to COM12


In [ ]:
ic.i2c_scan()

In [ ]:
ic.m3_init_ref(op_mode=4)

In [ ]:
ic.IIN_UCP_DIS = 1

In [ ]:
ic.status

In [ ]:
ic.enable_vin_charging

In [ ]:
ic.disable_charging

In [ ]:
c1 = "VOUT"
c2 = "C1NA"
c3 = "C2PA"
c4 = "C2NA"

for n in range(1, 2):
    ds.single
    delay(1.5)
    ic.enable_vin_charging
    delay(1)
    print(f"switching status : {ic.CP_SWITCHING_STAT}")
    # ds.save_waveform = f"[SC8583 CFLY_SHORT] {c1} 4v offset, {c2}, {c3} 3.5v offset, {c4} - 1kh 140mV pkpk waveform #{n} (switching={ic.CP_SWITCHING_STAT})"
    ic.disable_charging
    delay(1)

In [3]:
c1 = "C1NA"
c2 = "C2NA"
c3 = "C1NB"
c4 = "C2NB"

ds.ch1.vertical_scale_grid = 0.1, 1
ds.ch2.vertical_scale_grid = 0.1, -2
ds.ch3.vertical_scale_grid = 0.1, 0
ds.ch4.vertical_scale_grid = 0.1, -3
ds.ch1.bw_20MHz
ds.ch2.bw_20MHz
ds.ch3.bw_20MHz
ds.ch4.bw_20MHz

# ds.ch4.label = f"{c1}, {c2}, {c3}, {c4}"

ds.ch1.label = c1
ds.ch2.label = c2
ds.ch3.label = c3
ds.ch4.label = c4

In [6]:
ds.save_waveform = f"[SC8583 CFLY_SHORT IC Swap] {c1} {c2} {c3} {c4} soc 34 #13"

In [ ]:
ds.save_waveform = f"[SC8583 CFLY_SHORT] {c1} 4v offset, {c2}, {c3} 3.5v offset, {c4} - 500hz 120mV pkpk waveform #cursor 2"

In [ ]:
ic.status

In [ ]:
ic.ADC_EN = 1
ic.status_adc

In [ ]:
pz.init_powerz

pps_temp = 8.0
vout = 4.0

pz.cfg_all = pps_temp, 3
delay(1.5)
ret_vbus = pz.voltage
print(f"initial power-z voltage : {ret_vbus:.03f}")

while True:
    ret_vbus = pz.voltage
    if ret_vbus < vout*2.02:
        pps_temp += 0.02
        pz.cfg_all = pps_temp, 3
        print(f"PPS={pps_temp:.03f}  RET={ret_vbus:.03f}  DIFF={pps_temp-ret_vbus:.03f}")
        delay(0.2)
    else:
        print("break #1")
        break
    if pps_temp > 9.5:
        print("break #2")
        break

import numpy as np
vbus_list = list(np.arange(pps_temp, 9.51, 0.02))

In [ ]:
pz.init_powerz
pz.cfg_all = 8.3, 3

In [ ]:
ic.m3_init_ref(op_mode=2)
ic.IIN_UCP_DIS = 1
ic.ADC_EN = 1
ic.ADC_RATE = 0

In [ ]:
ic.enable_vin_charging

In [ ]:
log.output_set_filename(title="[MAXTIL] PD TA + Power-Z")
log.output_csv(message=["PPS (V)", "VIN (V)", "IIN (A)", "IIN_ADC (A)", "Diff (A)"])

for v in vbus_list:
    vbus = round(v, 3)
    pz.cfg_all = vbus, 3
    delay(0.5)
    cp_vbus = round(ic.vin_adc, 3)
    cp_iin = round(ic.iin_adc, 3)
    pz_iin = round(pz.current, 3)
    diff   = round((pz_iin - cp_iin), 3)
    print(f"VBUS={vbus:.03f}  VIN={cp_vbus:.03f}  Power-Z_IIN={pz_iin:.03f}  SC8583_IIN={cp_iin:.03f}  -->  Diff={pz_iin-cp_iin:.03f}")
    log.output_csv(message=[vbus, cp_vbus, cp_iin, pz_iin, diff])

    if cp_iin > 3: break

pz.cfg_all = pps_temp, 2

In [ ]:
iteration_vbus = [8.64, 8.66, 8.68, 8.7, 8.72, 8.74, 8.76, 8.78, 8.8, 8.82, 8.84, 8.86, 8.88, 8.9, 8.92, 8.94, 8.96, 8.98, 9, 9.02, 9.04, 9.06, 9.08, 9.1, 9.12, 9.14]
log.output_set_filename(title="[MAXTIL] PD TA + Power-Z, 500mA 100Hz load")
# log.output_csv(message=["N", "CP", "PPS (V)", "VIN (V)", "IIN (A)", "IIN_ADC (A)", "Diff (V)", "Diff (A)"])

for n in range(1, 2):

    for v in iteration_vbus:
        vbus = round(v, 3)
        pz.cfg_all = vbus, 3
        delay(0.15)
        cp_vbus = round(ic.vin_adc, 3)
        cp_iin = round(ic.iin_adc, 3)
        pz_iin = round(pz.current, 3)
        diff_v = round(vbus - cp_vbus, 3)
        diff_i   = round((pz_iin - cp_iin), 3)
        sw_stat = ic.CP_SWITCHING_STAT
        print(f"[{n}] CP={sw_stat} VBUS={vbus:.03f}  VIN={cp_vbus:.03f}  Power-Z_IIN={pz_iin:.03f}  SC8583_IIN={cp_iin:.03f}  -->  Diff={pz_iin-cp_iin:.03f}")
        log.output_csv(message=[n, sw_stat, vbus, cp_vbus, cp_iin, pz_iin, diff_v, diff_i])

        if cp_iin > 3: break
        if sw_stat != 1: break

    pz.cfg_all = iteration_vbus[0], 2
    delay(1)
    print(f"------------------------------------------------------------------------------------")

In [ ]:
import numpy as np
# iteration_vbus = [8.64, 8.66, 8.68, 8.7, 8.72, 8.74, 8.76, 8.78, 8.8, 8.82, 8.84, 8.86, 8.88, 8.9, 8.92, 8.94, 8.96, 8.98, 9, 9.02, 9.04, 9.06, 9.08, 9.1, 9.12, 9.14]
iteration_vbus = list(np.arange(8.5, 9.301, 0.04))
log.output_set_filename(title="[MAXTIL] PD TA + Power-Z, 100mA 100Hz 25p 200ms delay load")
log.output_csv(message=["N", "CP", "PPS (V)", "VIN (V)", "IIN_ADC (A)"])

for n in range(101):

    for v in iteration_vbus:
        vbus = round(v, 3)
        pz.cfg_all = vbus, 3
        delay(0.2)
        cp_vbus = round(ic.vin_adc, 3)
        cp_iin = round(ic.iin_adc, 3)
        # pz_iin = round(pz.current, 3)
        # diff_v = round(vbus - cp_vbus, 3)
        # diff_i   = round((pz_iin - cp_iin), 3)
        sw_stat = ic.CP_SWITCHING_STAT
        print(f"[{n}] PPS={vbus:.03f}  CP={sw_stat}  VIN={cp_vbus:.03f}  SC8583_IIN={cp_iin:.03f}")
        log.output_csv(message=[n, sw_stat, vbus, cp_vbus, cp_iin])

        if cp_iin > 3: break
        if sw_stat != 1: break

    pz.cfg_all = iteration_vbus[0], 2
    delay(1)
    print(f"-------------------------------------")

In [ ]:
ic.disable_charging

In [ ]:
pz.cfg_all = 8.6, 3

In [ ]:
ic.status